In [1]:
# ***CV PROJECT***
# Cell 1: Imports, Configuration, Kaggle Download, and Data Loadingggg

import os
import json
import zipfile
import numpy as np
import pandas as pd
from PIL import Image
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.transforms as T

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import warnings
warnings.filterwarnings("ignore")

# — Kaggle credentials & download —
creds = {"username":"sunil22516","key":"469344f061af16f02b69add143462a17"}
cfg = os.path.expanduser("~/.config/kaggle")
os.makedirs(cfg, exist_ok=True)
with open(os.path.join(cfg,"kaggle.json"), "w") as f:
    json.dump(creds, f)
os.chmod(os.path.join(cfg,"kaggle.json"), 0o600)

from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_files('visual-taxonomy', path='data', quiet=False)
with zipfile.ZipFile('data/visual-taxonomy.zip','r') as z:
    z.extractall('data')




TRAIN_CSV    = "data/train.csv"
IMAGE_DIR    = "data/train_images"
MODEL_PATH   = "best_model.pth"
IMG_SIZE     = 224
BATCH_SIZE   = 16
NUM_EPOCHS   = 5
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")


df_listed = pd.read_csv(TRAIN_CSV)
attr_cols = [f"attr_{i}" for i in range(1, 11)]
print(f"Loaded {len(df_listed)} samples from {TRAIN_CSV}")

# # Cell 1: Kaggle Authentication via Environment Variables
# import os

# # Set your Kaggle API credentials
# os.environ['KAGGLE_USERNAME'] = 'sunil22516'
# os.environ['KAGGLE_KEY'] = '469344f061af16f02b69add143462a17'

# # Install and configure Kaggle CLI (if not already installed)
# !pip install kaggle --quiet
# !mkdir -p ~/.config/kaggle

# # No need to copy kaggle.json; environment variables suffice for authentication

# # Cell 2: Download and Unzip the Dataset
# from kaggle.api.kaggle_api_extended import KaggleApi

# api = KaggleApi()
# api.authenticate()

# # Download the dataset (utkarsh1232/attribute-extraction-meesho) into `data/`
# !mkdir -p data
# api.dataset_download_files('utkarsh1232/attribute-extraction-meesho', path='data', unzip=True, force=True)

# # Verify files
# print("Contents of data/:")
# !ls -R data


# # Cell 3 onward: (Pipeline continues as previously defined)
# # Cell 3 onward: (Pipeline continues as previously defined)

visual-taxonomy.zip: Skipping, found more recently modified local copy (use --force to force download)
Loaded 70213 samples from data/train.csv


In [2]:
# Cell 2: Label Encoding, Stratified Split, Balanced Sampling & DataLoaders

import os
import numpy as np
from collections import Counter
from PIL import Image
from datetime import datetime

from torch.utils.data import WeightedRandomSampler, DataLoader, Dataset
import torchvision.transforms as T
from torchvision.transforms import RandAugment, RandomResizedCrop, RandomErasing

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedShuffleSplit

# 1. Define and fit per-attribute label encoders (with inverse helper)
class LabelEncoderDict:
    def __init__(self):
        self.encoders = {}

    def fit(self, df, cols):
        for c in cols:
            le = LabelEncoder()
            le.fit(df[c].fillna("NaN").astype(str).unique())
            self.encoders[c] = le

    def transform(self, df_rows, cols):
        arr = np.zeros((len(df_rows), len(cols)), dtype=int)
        for i, c in enumerate(cols):
            arr[:, i] = self.encoders[c].transform(
                df_rows[c].fillna("NaN").astype(str)
            )
        return arr

    def inverse(self, idx, col):
        # idx: integer class index, col: attribute name
        return self.encoders[col].inverse_transform([idx])[0]

led = LabelEncoderDict()
led.fit(df_listed, attr_cols)

# Number of classes per attribute
num_classes_per_attr = [len(led.encoders[c].classes_) for c in attr_cols]

# 2. Stratified train/validation split on the most imbalanced attribute (attr_1)
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
y_strat = df_listed['attr_1'].fillna("NaN")
for tr_idx, val_idx in splitter.split(df_listed, y_strat):
    train_part = df_listed.iloc[tr_idx].reset_index(drop=True)
    val_part   = df_listed.iloc[val_idx].reset_index(drop=True)

# 3. Compute sample weights to oversample rare labels in train_part
label_counts = {c: Counter(train_part[c].fillna("NaN")) for c in attr_cols}
def sample_weight(row):
    return sum(1.0 / label_counts[c][row[c]] for c in attr_cols if row[c] == row[c])
train_weights = train_part.apply(sample_weight, axis=1).values
sampler = WeightedRandomSampler(train_weights, num_samples=len(train_weights), replacement=True)

# 4. Define strong augmentations for training and standard for validation
train_tf = T.Compose([
    RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    RandAugment(num_ops=4, magnitude=7),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(0.2, 0.2, 0.2, 0.05),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
    RandomErasing(p=0.3, scale=(0.02, 0.1))
])
val_tf = T.Compose([
    T.Resize(IMG_SIZE),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])

# 5. Dataset class applying transforms
class SimpleDS(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(IMAGE_DIR, f"{row['id']:06d}.jpg")).convert("RGB")
        x = self.transform(img)
        y = led.transform(self.df.iloc[[idx]], attr_cols)[0]
        return x, y

# 6. Create DataLoaders
train_ds = SimpleDS(train_part, train_tf)
val_ds   = SimpleDS(val_part,   val_tf)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=0,
    pin_memory=True
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [3]:
# Cell 3: Focal Loss Definition (to emphasize hard & rare examples)

import torch
import torch.nn as nn

class FocalLoss(nn.Module):
    """
    Multi-head focal loss for multi-label classification.
    Focuses training on hard, misclassified examples, with per-class weighting.
    """
    def __init__(self, class_weights, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        # list of 1D tensors, one weight tensor per attribute head
        self.class_weights = class_weights
        # Use reduction='none' so we can apply focal manually
        self.ce = nn.CrossEntropyLoss(weight=None, reduction='none')

    def forward(self, outputs, targets):
        """
        outputs: list of [batch_size x num_classes_i] logits, one per attribute
        targets: LongTensor [batch_size x num_attributes] ground-truth indices
        """
        total_loss = 0.0
        for i, out in enumerate(outputs):
            # compute per-sample cross-entropy
            ce_loss = self.ce(out, targets[:, i])            # [batch_size]
            # pt = probability of the true class
            pt = torch.exp(-ce_loss)                         # [batch_size]
            # per-sample weight = class_weight for the true class
            w = self.class_weights[i][targets[:, i]]         # [batch_size]
            # focal scaling
            focal_term = (1 - pt) ** self.gamma              # [batch_size]
            loss_i = w * focal_term * ce_loss                # [batch_size]
            total_loss += loss_i.mean()                      # scalar
        # average over all attribute heads
        return total_loss / len(outputs)


# — Compute per-class weights for each attribute head —
class_weights = []
for c in attr_cols:
    freqs = df_listed[c].fillna("NaN").value_counts(normalize=True)
    # inverse frequency as weight
    w = torch.tensor((1.0 / freqs).values, dtype=torch.float32).to(DEVICE)
    class_weights.append(w)


class BalancedSoftmaxLoss(nn.Module):
    def __init__(self, class_counts):
        super().__init__()
        # compute log prior for each head
        priors = []
        for cnts in class_counts:
            p = torch.tensor(cnts, dtype=torch.float32)
            p = p / p.sum()
            priors.append(p.log().to(DEVICE))
        self.log_priors = nn.ParameterList([nn.Parameter(lp, requires_grad=False) for lp in priors])
    def forward(self, outputs, targets):
        loss = 0.0
        for i, out in enumerate(outputs):
            # shift logits by log prior
            out_adj = out + self.log_priors[i].unsqueeze(0)
            loss += F.cross_entropy(out_adj, targets[:,i])
        return loss / len(outputs)

# compute raw counts per head
counts = [df_listed[c].fillna("NaN").value_counts().sort_index().values for c in attr_cols]
criterion = BalancedSoftmaxLoss(counts)

In [4]:
# Testing out models on 10 percent of data
# Cell X: Model Definition & Training Loop with Gradual Unfreeze, MixUp, Focal Loss, Early Stopping (using 10 percent )

import os
import numpy as np
import torch
import torch.nn as nn
import timm
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.metrics import f1_score

# ─── Sanity checks ────────────────────────────────────────────────────────────
# Make sure you've run Cell 1 (defines DEVICE, MODEL_PATH) and Cell 2 (defines led, train_loader, val_loader, attr_cols)
assert 'DEVICE' in globals(), "Run Cell 1 first (defines DEVICE)."
assert 'led' in globals(),       "Run Cell 2 first (defines led)."
assert 'train_loader' in globals() and 'val_loader' in globals(), "Run Cell 2 first (defines data loaders)."

# ─── Hyperparameters ─────────────────────────────────────────────────────────
NUM_EPOCHS = 10
PATIENCE   = 5
REDUCE_SIZE = 0.1  # Reduce dataset to 10% of original

# ─── Reduce dataset size ─────────────────────────────────────────────────────
train_subset_size = int(len(train_loader.dataset) * REDUCE_SIZE)
val_subset_size = int(len(val_loader.dataset) * REDUCE_SIZE)

# Create subsets of the data for training and validation
train_subset = torch.utils.data.Subset(train_loader.dataset, range(train_subset_size))
val_subset = torch.utils.data.Subset(val_loader.dataset, range(val_subset_size))

# Create new DataLoaders for the reduced dataset
train_loader = torch.utils.data.DataLoader(train_subset, batch_size=16, shuffle=True, num_workers=0)
val_loader = torch.utils.data.DataLoader(val_subset, batch_size=16, shuffle=False, num_workers=0)

# ─── Compute number of classes per attribute ─────────────────────────────────
num_classes_per_attr = [len(led.encoders[c].classes_) for c in attr_cols]

# ─── Model architecture ───────────────────────────────────────────────────────
class MultiLabelClassifier(nn.Module):
    """
    Multi-label classifier using timm's EfficientNet-B5 backbone
    with one separate head per attribute.
    """
    def __init__(self, num_classes_per_attr):
        super().__init__()
        # Use EfficientNet-B5 (larger than B0)
        self.backbone = timm.create_model(
            'efficientnet_b5',
            pretrained=True,
            num_classes=0,      # remove original head
            global_pool='avg'
        )
        feat_dim = self.backbone.num_features

        # One MLP head per attribute
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(feat_dim, 512),
                nn.ReLU(inplace=True),
                nn.Dropout(0.3),
                nn.Linear(512, n)
            )
            for n in num_classes_per_attr
        ])

    def forward(self, x):
        features = self.backbone(x)           # [batch_size, feat_dim]
        return [head(features) for head in self.heads]

# ─── Instantiate model ───────────────────────────────────────────────────────
model = MultiLabelClassifier(num_classes_per_attr).to(DEVICE)

# ─── Optimizer, Scheduler, Mixed-Precision Scaler ────────────────────────────
optimizer = optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': 1e-5},
    {'params': model.heads.parameters(),      'lr': 1e-4}
], weight_decay=1e-3)

total_steps = NUM_EPOCHS * len(train_loader)
scheduler   = OneCycleLR(optimizer, max_lr=5e-4, total_steps=total_steps)
scaler      = GradScaler()

# ─── Focal Loss Definition ───────────────────────────────────────────────────
class FocalLoss(nn.Module):
    """
    Multi-head focal loss to handle class imbalance
    """
    def __init__(self, class_weights, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.class_weights = class_weights
        self.ce = nn.CrossEntropyLoss(weight=None, reduction='none')

    def forward(self, outputs, targets):
        total_loss = 0.0
        for i, out in enumerate(outputs):
            ce_loss = self.ce(out, targets[:, i])      # [batch_size]
            pt      = torch.exp(-ce_loss)               # [batch_size]
            w       = self.class_weights[i][targets[:, i]]  # [batch_size]
            focal   = w * ((1 - pt) ** self.gamma) * ce_loss
            total_loss += focal.mean()
        return total_loss / len(outputs)

# ─── Compute per-class weights for FocalLoss ─────────────────────────────────
class_weights = []
for c in attr_cols:
    freqs = df_listed[c].fillna("NaN").value_counts(normalize=True)
    w = torch.tensor((1.0 / freqs).values, dtype=torch.float32).to(DEVICE)
    class_weights.append(w)

criterion = FocalLoss(class_weights, gamma=2.0)

# ─── Training Loop with Gradual Unfreeze & Early Stopping ────────────────────
best_macro_f1, patience = 0.0, 0

for epoch in range(1, NUM_EPOCHS + 1):
    # ─── Gradual Unfreeze Schedule ───────────────────────────────────────────
    if epoch == 1:
        # freeze entire backbone for first epoch
        for p in model.backbone.parameters():
            p.requires_grad = False
    if epoch == 4:
        # unfreeze last two blocks (timm names them differently; adjust if needed)
        for name, p in model.backbone.named_parameters():
            if 'blocks.6' in name or 'blocks.7' in name:
                p.requires_grad = True
    if epoch == 8:
        # unfreeze full backbone
        for p in model.backbone.parameters():
            p.requires_grad = True

    model.train()
    running_loss = 0.0

    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        # MixUp augmentation
        lam = np.random.beta(0.2, 0.2)
        idx = torch.randperm(x.size(0))
        x_mix = lam * x + (1 - lam) * x[idx]
        y_a, y_b = y, y[idx]

        optimizer.zero_grad()
        with autocast():
            outs = model(x_mix)
            loss = lam * criterion(outs, y_a) + (1 - lam) * criterion(outs, y_b)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item() * x.size(0)

    train_loss = running_loss / len(train_loader.dataset)

    # ─── Validation: compute Macro-F1 ────────────────────────────────────────
    model.eval()
    all_preds, all_trues = [], []
    with torch.no_grad():
        for x_val, y_val in val_loader:
            x_val, y_val = x_val.to(DEVICE), y_val.to(DEVICE)
            outs_val = model(x_val)
            preds = torch.stack([o.argmax(1) for o in outs_val], dim=1)  # [B, heads]
            all_preds.append(preds.cpu().numpy())
            all_trues.append(y_val.cpu().numpy())

    all_preds = np.vstack(all_preds)
    all_trues = np.vstack(all_trues)
    macro_f1 = np.mean([f1_score(all_trues[:, i], all_preds[:, i], average='macro') for i in range(all_trues.shape[1])])

    print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, Val Macro-F1 = {macro_f1:.4f}")

    # ─── Early Stopping & Checkpointing ─────────────────────────────────────
    if macro_f1 > best_macro_f1:
        best_macro_f1, patience = macro_f1, 0
        torch.save(model.state_dict(), MODEL_PATH)
        print("  → New best Macro-F1, model saved")
    else:
        patience += 1
        if patience >= PATIENCE:
            print(f"  Early stopping triggered (no improvement in {PATIENCE} epochs).")
            break

Epoch 1: Train Loss = 73.0960, Val Macro-F1 = 0.1985
  → New best Macro-F1, model saved
Epoch 2: Train Loss = 35.4591, Val Macro-F1 = 0.2688
  → New best Macro-F1, model saved


KeyboardInterrupt: 

In [ ]:
# Cell X: EfficientNet-B5 Backbone Model Definition & Training Loop (Reduced Dataset)

import os
import numpy as np
import torch
import torch.nn as nn
import timm
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.metrics import f1_score

# ── Sanity checks ─────────────────────────────────────────────────────────────
assert 'DEVICE'       in globals(), "Run Cell 1 first (defines DEVICE)."
assert 'train_loader' in globals() and 'val_loader' in globals(), "Run Cell 2 first (defines DataLoaders)."
assert 'criterion'    in globals(), "Run Cell 3 first (defines loss)."
assert 'attr_cols'    in globals(), "Run Cell 1/2 first (defines attr_cols)."
assert 'NUM_EPOCHS'   in globals(), "Define NUM_EPOCHS in this cell or above."
assert 'PATIENCE'     in globals(), "Define PATIENCE in this cell or above."

# ── Hyperparameters ───────────────────────────────────────────────────────────
NUM_EPOCHS   = 10    # total epochs
PATIENCE     = 5     # early stopping patience
REDUCE_RATIO = 0.1   # use only 10% of data

# ── Reduce dataset size ───────────────────────────────────────────────────────
train_size = int(len(train_loader.dataset) * REDUCE_RATIO)
val_size   = int(len(val_loader.dataset)   * REDUCE_RATIO)

train_subset = torch.utils.data.Subset(train_loader.dataset, range(train_size))
val_subset   = torch.utils.data.Subset(val_loader.dataset,   range(val_size))

train_loader = torch.utils.data.DataLoader(
    train_subset,
    batch_size=16,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)
val_loader = torch.utils.data.DataLoader(
    val_subset,
    batch_size=16,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

# ── Compute number of classes per attribute ───────────────────────────────────
num_classes_per_attr = [len(led.encoders[c].classes_) for c in attr_cols]

# ── Model architecture using EfficientNet-B5 (Noisy Student) ─────────────────
class MultiLabelEffNetB5(nn.Module):
    def __init__(self, num_classes_per_attr):
        super().__init__()
        self.backbone = timm.create_model(
            'tf_efficientnet_b5_ns',
            pretrained=True,
            num_classes=0,
            global_pool='avg'
        )
        feat_dim = self.backbone.num_features
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(feat_dim, 512),
                nn.ReLU(inplace=True),
                nn.Dropout(0.3),
                nn.Linear(512, n)
            ) for n in num_classes_per_attr
        ])

    def forward(self, x):
        feats = self.backbone(x)               # [B, feat_dim]
        return [h(feats) for h in self.heads]

# ── Instantiate model ─────────────────────────────────────────────────────────
model = MultiLabelEffNetB5(num_classes_per_attr).to(DEVICE)

# ── Optimizer & Scheduler ────────────────────────────────────────────────────
optimizer = optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': 1e-5},
    {'params': model.heads.parameters(),      'lr': 1e-4}
], weight_decay=1e-3)

total_steps = NUM_EPOCHS * len(train_loader)
scheduler   = OneCycleLR(optimizer, max_lr=5e-4, total_steps=total_steps)
scaler      = GradScaler()

# ── Training Loop with Gradual Unfreeze, MixUp & Early Stopping ─────────────
best_macro_f1, patience = 0.0, 0

for epoch in range(1, NUM_EPOCHS + 1):
    # Gradual unfreeze schedule
    if epoch == 1:
        for p in model.backbone.parameters(): p.requires_grad = False
    if epoch == 4:
        for name, p in model.backbone.named_parameters():
            if 'blocks.6' in name or 'blocks.7' in name:
                p.requires_grad = True
    if epoch == 8:
        for p in model.backbone.parameters(): p.requires_grad = True

    model.train()
    running_loss = 0.0

    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        lam = np.random.beta(0.2, 0.2)
        idx = torch.randperm(x.size(0))
        x_mix = lam * x + (1 - lam) * x[idx]
        y_a, y_b = y, y[idx]

        optimizer.zero_grad()
        with autocast():
            outs = model(x_mix)
            loss = lam * criterion(outs, y_a) + (1 - lam) * criterion(outs, y_b)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item() * x.size(0)

    train_loss = running_loss / len(train_loader.dataset)

    # Validation: compute Macro-F1
    model.eval()
    all_preds, all_trues = [], []
    with torch.no_grad():
        for x_val, y_val in val_loader:
            x_val, y_val = x_val.to(DEVICE), y_val.to(DEVICE)
            outs_val = model(x_val)
            preds = torch.stack([o.argmax(1) for o in outs_val], dim=1)
            all_preds.append(preds.cpu().numpy())
            all_trues.append(y_val.cpu().numpy())

    all_preds = np.vstack(all_preds)
    all_trues = np.vstack(all_trues)
    macro_f1 = np.mean([
        f1_score(all_trues[:, i], all_preds[:, i], average='macro')
        for i in range(all_trues.shape[1])
    ])

    print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, Val Macro-F1 = {macro_f1:.4f}")

    # Early stopping & checkpointing
    if macro_f1 > best_macro_f1:
        best_macro_f1, patience = macro_f1, 0
        torch.save(model.state_dict(), MODEL_PATH)
        print("  → New best Macro-F1, model saved")
    else:
        patience += 1
        if patience >= PATIENCE:
            print(f"  Early stopping (no improvement in {PATIENCE} epochs).")
            break

In [ ]:
# Cell X: Swin Transformer Backbone (Swin-B) Model Definition & Training Loop

import os
import numpy as np
import torch
import torch.nn as nn
import timm
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import Subset, DataLoader
from sklearn.metrics import f1_score

# ── Sanity checks ─────────────────────────────────────────────────────────────
assert 'DEVICE'    in globals(), "Run Cell 1 first (defines DEVICE)."
assert 'train_ds'  in globals() and 'val_ds' in globals(), "Run Cell 2 first (defines train_ds/val_ds)."
assert 'criterion' in globals(), "Run Cell 3 first (defines criterion)."
assert 'attr_cols' in globals(), "Run Cell 1/2 first (defines attr_cols)."

# ── Hyperparameters ───────────────────────────────────────────────────────────
NUM_EPOCHS   = 10
PATIENCE     = 5
REDUCE_RATIO = 0.1   # use 10% of the data for faster iteration
BATCH_SIZE   = 16

# ── Create smaller subsets to speed up training ───────────────────────────────
train_size = int(len(train_ds) * REDUCE_RATIO)
val_size   = int(len(val_ds)   * REDUCE_RATIO)

train_subset       = Subset(train_ds, range(train_size))
val_subset         = Subset(val_ds,   range(val_size))

train_loader_swin  = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,
                                num_workers=4, pin_memory=True)
val_loader_swin    = DataLoader(val_subset,   batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=4, pin_memory=True)

# ── Compute number of classes per attribute ───────────────────────────────────
num_classes_per_attr = [len(led.encoders[c].classes_) for c in attr_cols]

# ── Model architecture using Swin-B (patch4_window7_224) ─────────────────────
class MultiLabelSwinB(nn.Module):
    def __init__(self, num_classes_per_attr):
        super().__init__()
        self.backbone = timm.create_model(
            'swin_base_patch4_window7_224',
            pretrained=True,
            num_classes=0,
            global_pool='avg'
        )
        feat_dim = self.backbone.num_features
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(feat_dim, 512),
                nn.ReLU(inplace=True),
                nn.Dropout(0.3),
                nn.Linear(512, n)
            ) for n in num_classes_per_attr
        ])

    def forward(self, x):
        feats = self.backbone(x)                   # [B, feat_dim]
        return [h(feats) for h in self.heads]

# ── Instantiate model ─────────────────────────────────────────────────────────
model = MultiLabelSwinB(num_classes_per_attr).to(DEVICE)

# ── Optimizer & Scheduler ────────────────────────────────────────────────────
optimizer = optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': 1e-5},
    {'params': model.heads.parameters(),      'lr': 1e-4}
], weight_decay=1e-3)

total_steps = NUM_EPOCHS * len(train_loader_swin)
scheduler   = OneCycleLR(optimizer, max_lr=5e-4, total_steps=total_steps)
scaler      = GradScaler()

# ── Training Loop with MixUp, Gradual Unfreeze & Early Stopping ─────────────
best_macro_f1, patience = 0.0, 0

for epoch in range(1, NUM_EPOCHS+1):
    # Gradual unfreeze schedule
    if epoch == 1:
        for p in model.backbone.parameters(): p.requires_grad = False
    if epoch == 4:
        for p in model.backbone.parameters(): p.requires_grad = True

    model.train()
    running_loss = 0.0

    for x, y in train_loader_swin:
        x, y = x.to(DEVICE), y.to(DEVICE)
        lam = np.random.beta(0.2, 0.2)
        idx = torch.randperm(x.size(0))
        x_mix = lam * x + (1 - lam) * x[idx]
        y_a, y_b = y, y[idx]

        optimizer.zero_grad()
        with autocast():
            outs = model(x_mix)
            loss = lam * criterion(outs, y_a) + (1 - lam) * criterion(outs, y_b)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running_loss += loss.item() * x.size(0)

    train_loss = running_loss / len(train_loader_swin.dataset)

    # Validation: compute per-head macro-F1
    model.eval()
    all_preds, all_trues = [], []
    with torch.no_grad():
        for x_val, y_val in val_loader_swin:
            x_val, y_val = x_val.to(DEVICE), y_val.to(DEVICE)
            outs_val = model(x_val)
            preds = torch.stack([o.argmax(1) for o in outs_val], dim=1)
            all_preds.append(preds.cpu().numpy())
            all_trues.append(y_val.cpu().numpy())

    all_preds = np.vstack(all_preds)
    all_trues = np.vstack(all_trues)
    macro_f1 = np.mean([
        f1_score(all_trues[:, i], all_preds[:, i], average='macro')
        for i in range(all_trues.shape[1])
    ])

    print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, Val Macro-F1 = {macro_f1:.4f}")

    # Early stopping & checkpointing
    if macro_f1 > best_macro_f1:
        best_macro_f1, patience = macro_f1, 0
        torch.save(model.state_dict(), MODEL_PATH)
        print("  → New best Macro-F1, model saved")
    else:
        patience += 1
        if patience >= PATIENCE:
            print(f"  Early stopping (no improvement in {PATIENCE} epochs).")
            break

In [ ]:

# Cell 4: Model Definition & Training Loop with Gradual Unfreeze, MixUp, Focal Loss, Early Stopping

import os
import numpy as np
import torch
import torch.nn as nn
import timm
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.metrics import f1_score

# ─── Sanity checks ────────────────────────────────────────────────────────────
# Make sure you've run Cell 1 (defines DEVICE, MODEL_PATH) and Cell 2 (defines led, train_loader, val_loader, attr_cols)
assert 'DEVICE' in globals(), "Run Cell 1 first (defines DEVICE)."
assert 'led' in globals(),       "Run Cell 2 first (defines led)."
assert 'train_loader' in globals() and 'val_loader' in globals(), "Run Cell 2 first (defines data loaders)."

# ─── Hyperparameters ─────────────────────────────────────────────────────────
NUM_EPOCHS = 10
PATIENCE   = 5

# ─── Compute number of classes per attribute ─────────────────────────────────
num_classes_per_attr = [len(led.encoders[c].classes_) for c in attr_cols]

# ─── Model architecture ───────────────────────────────────────────────────────
class MultiLabelClassifier(nn.Module):
    """
    Multi-label classifier using timm's EfficientNet-B3 backbone
    with one separate head per attribute.
    """
    def __init__(self, num_classes_per_attr):
        super().__init__()
        # Use EfficientNet-B3 (larger than B0)
        self.backbone = timm.create_model(
            'efficientnet_b5',
            pretrained=True,
            num_classes=0,      # remove original head
            global_pool='avg'
        )
        feat_dim = self.backbone.num_features

        # One MLP head per attribute
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(feat_dim, 512),
                nn.ReLU(inplace=True),
                nn.Dropout(0.3),
                nn.Linear(512, n)
            )
            for n in num_classes_per_attr
        ])

    def forward(self, x):
        features = self.backbone(x)           # [batch_size, feat_dim]
        return [head(features) for head in self.heads]

# ─── Instantiate model ───────────────────────────────────────────────────────
model = MultiLabelClassifier(num_classes_per_attr).to(DEVICE)

# ─── Optimizer, Scheduler, Mixed-Precision Scaler ────────────────────────────
optimizer = optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': 1e-5},
    {'params': model.heads.parameters(),      'lr': 1e-4}
], weight_decay=1e-3)

total_steps = NUM_EPOCHS * len(train_loader)
scheduler   = OneCycleLR(optimizer, max_lr=5e-4, total_steps=total_steps)
scaler      = GradScaler()

# ─── Focal Loss Definition ───────────────────────────────────────────────────
class FocalLoss(nn.Module):
    """
    Multi-head focal loss to handle class imbalance
    """
    def __init__(self, class_weights, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.class_weights = class_weights
        self.ce = nn.CrossEntropyLoss(weight=None, reduction='none')

    def forward(self, outputs, targets):
        total_loss = 0.0
        for i, out in enumerate(outputs):
            ce_loss = self.ce(out, targets[:, i])      # [batch_size]
            pt      = torch.exp(-ce_loss)               # [batch_size]
            w       = self.class_weights[i][targets[:, i]]  # [batch_size]
            focal   = w * ((1 - pt) ** self.gamma) * ce_loss
            total_loss += focal.mean()
        return total_loss / len(outputs)

# ─── Compute per-class weights for FocalLoss ─────────────────────────────────
class_weights = []
for c in attr_cols:
    freqs = df_listed[c].fillna("NaN").value_counts(normalize=True)
    w = torch.tensor((1.0 / freqs).values, dtype=torch.float32).to(DEVICE)
    class_weights.append(w)

criterion = FocalLoss(class_weights, gamma=2.0)

# ─── Training Loop with Gradual Unfreeze & Early Stopping ────────────────────
best_macro_f1, patience = 0.0, 0

for epoch in range(1, NUM_EPOCHS + 1):
    # ─── Gradual Unfreeze Schedule ───────────────────────────────────────────
    if epoch == 1:
        # freeze entire backbone for first epoch
        for p in model.backbone.parameters():
            p.requires_grad = False
    if epoch == 4:
        # unfreeze last two blocks (timm names them differently; adjust if needed)
        for name, p in model.backbone.named_parameters():
            if 'blocks.6' in name or 'blocks.7' in name:
                p.requires_grad = True
    if epoch == 8:
        # unfreeze full backbone
        for p in model.backbone.parameters():
            p.requires_grad = True

    model.train()
    running_loss = 0.0

    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        # MixUp augmentation
        lam = np.random.beta(0.2, 0.2)
        idx = torch.randperm(x.size(0))
        x_mix = lam * x + (1 - lam) * x[idx]
        y_a, y_b = y, y[idx]

        optimizer.zero_grad()
        with autocast():
            outs = model(x_mix)
            loss = lam * criterion(outs, y_a) + (1 - lam) * criterion(outs, y_b)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item() * x.size(0)

    train_loss = running_loss / len(train_loader.dataset)

    # ─── Validation: compute Macro-F1 ────────────────────────────────────────
    model.eval()
    all_preds, all_trues = [], []
    with torch.no_grad():
        for x_val, y_val in val_loader:
            x_val, y_val = x_val.to(DEVICE), y_val.to(DEVICE)
            outs_val = model(x_val)
            preds = torch.stack([o.argmax(1) for o in outs_val], dim=1)  # [B, heads]
            all_preds.append(preds.cpu().numpy())
            all_trues.append(y_val.cpu().numpy())

    all_preds = np.vstack(all_preds)
    all_trues = np.vstack(all_trues)
    macro_f1 = np.mean([
        f1_score(all_trues[:, i], all_preds[:, i], average='macro')
        for i in range(all_preds.shape[1])
    ])

    print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, Val Macro-F1 = {macro_f1:.4f}")

    # ─── Early Stopping & Checkpointing ─────────────────────────────────────
    if macro_f1 > best_macro_f1:
        best_macro_f1, patience = macro_f1, 0
        torch.save(model.state_dict(), MODEL_PATH)
        print("  → New best Macro-F1, model saved")
    else:
        patience += 1
        if patience >= PATIENCE:
            print(f"  Early stopping triggered (no improvement in {PATIENCE} epochs).")
            break

In [5]:
# Cell 6: Validation Accuracy & Confusion Matrices

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score

# 1. Compute per-attribute accuracy
correct = np.zeros(len(attr_cols), dtype=int)
total = 0

model.eval()
with torch.no_grad():
    for x, y in val_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        outs = model(x)
        for i, out in enumerate(outs):
            preds = out.argmax(dim=1)
            correct[i] += (preds == y[:, i]).sum().item()
        total += y.size(0)

print("=== Per-Attribute Accuracy ===")
for i, col in enumerate(attr_cols):
    acc = correct[i] / total * 100
    print(f"  {col}: {acc:.2f}%")

# 2. Gather all predictions and truths for confusion matrices
all_trues, all_preds = [], []
model.eval()
with torch.no_grad():
    for x, y in val_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        outs = model(x)
        preds = torch.stack([o.argmax(1) for o in outs], dim=1)
        all_preds.append(preds.cpu().numpy())
        all_trues.append(y.cpu().numpy())

all_preds = np.vstack(all_preds)
all_trues = np.vstack(all_trues)

# 3. Display confusion matrices for first 3 attributes
for i in range(min(3, len(attr_cols))):
    cm = confusion_matrix(all_trues[:, i], all_preds[:, i])
    disp = ConfusionMatrixDisplay(cm, display_labels=led.encoders[attr_cols[i]].classes_)
    fig, ax = plt.subplots(figsize=(6, 6))
    disp.plot(ax=ax, xticks_rotation='vertical')
    ax.set_title(f"Confusion Matrix for {attr_cols[i]}")
    plt.show()

# 4. Compute and print per-attribute Macro-F1
print("\n=== Per-Attribute Macro F1 ===")
for i, col in enumerate(attr_cols):
    f1 = f1_score(all_trues[:, i], all_preds[:, i], average='macro')
    print(f"  {col}: {f1:.3f}")

KeyboardInterrupt: 

In [ ]:
# Cell 7: Discrepancy Detection & Flagging

import os
import pandas as pd
import torch.nn.functional as F
from datetime import datetime
from PIL import Image

def analyze_image(image_path):
    img = Image.open(image_path).convert("RGB")
    x   = val_tf(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        outs = model(x)

    preds, confs = [], []
    for i, out in enumerate(outs):
        ps = F.softmax(out, dim=1).cpu().numpy()[0]
        idx = ps.argmax()
        # Use the new inverse helper
        preds.append(led.inverse(idx, attr_cols[i]))
        confs.append(ps[idx])
    return preds, confs

flags = []
for _, row in df_listed.iterrows():
    image_id = f"{row['id']:06d}"
    listed   = row[attr_cols].fillna("NaN").astype(str).tolist()
    img_path = os.path.join(IMAGE_DIR, f"{image_id}.jpg")
    predicted, confidence = analyze_image(img_path)

    disc = []
    for attr, l, p, c in zip(attr_cols, listed, predicted, confidence):
        if l != p:
            disc.append({
                "attribute": attr,
                "listed":     l,
                "predicted":  p,
                "confidence": round(float(c), 3)
            })
    if disc:
        flags.append({
            "image_id":    image_id,
            "discrepancies": disc,
            "timestamp":   datetime.utcnow().isoformat()
        })

# Flatten and save flags
out_rows = []
for f in flags:
    for d in f["discrepancies"]:
        out_rows.append([
            f["image_id"],
            d["attribute"],
            d["listed"],
            d["predicted"],
            d["confidence"],
            f["timestamp"]
        ])

flags_df = pd.DataFrame(
    out_rows,
    columns=["image_id","attribute","listed","predicted","confidence","timestamp"]
)
flags_df.to_csv("discrepancy_flags.csv", index=False)
print(f"Generated {len(flags_df)} discrepancy records.")

In [ ]:
# Cell 8: Feedback Message Generation & Summary (unchanged)

messages = []
for _, row in flags_df.iterrows():
    msg = (f"Image {row['image_id']}: Attribute '{row['attribute']}' mismatch — "
           f"listed='{row['listed']}', predicted='{row['predicted']}' "
           f"(conf={row['confidence']*100:.1f}%)")
    messages.append(msg)

with open("feedback_messages.txt", "w") as f:
    for m in messages:
        f.write(m + "\n")

print(f"\nTotal flagged images: {flags_df['image_id'].nunique()}/"
      f"{len(df_listed)} "
      f"({flags_df['image_id'].nunique()/len(df_listed)*100:.1f}%)")

print("\nSample feedback messages:")
for sample in messages[:5]:
    print(" •", sample)